# Week 2 Assessment - Indian Startup Funding, Decoded
**Name:** Y V S Harshith
**Course:** DADS - Week 2, Pandas + EDA
**Dataset:** indian_startups_funding.csv

This notebook covers Section A (theory), Section B (predict-the-output), and Section C (applied work on the Indian startup funding dataset).

## Section A - Interview Theory

In [1]:
# ========== Q1 ==========

**Q1. Series vs DataFrame**

A `Series` is a single one-dimensional labelled array, essentially one column with an index. A `DataFrame` is a two-dimensional table made of multiple Series that share the same index, so it behaves like a dictionary of columns. An operation that works on both is `.mean()` - on a Series it returns one number, on a DataFrame it returns a Series of column-wise means. An operation that only makes sense on a DataFrame is `.merge()`, since joining needs at least two columns of context (a key column plus the columns being brought in).

In [2]:
# ========== Q2 ==========

**Q2. inplace=True**

`inplace=True` makes a pandas method modify the calling object directly and return `None`, instead of returning a new object and leaving the original untouched. Newer pandas versions discourage it because it still creates a copy internally in most cases (so it rarely saves memory), it blocks method chaining, and it silently overwrites your original data, which makes debugging harder. I would avoid it when cleaning a dataset in a notebook, since I want to keep the raw `df` intact to re-check my cleaning steps if something looks wrong.

In [3]:
# ========== Q3 ==========

**Q3. .fillna() vs .dropna()**

`.fillna()` keeps every row and replaces missing values with something reasonable, like the column mean, a fixed value, or a forward-fill. `.dropna()` removes rows or columns that contain missing values entirely. I would use `.fillna()` when the missing values are a small share of a column and can be reasonably estimated, for example filling a missing `rating` with the column average so the rest of that row is not lost. I would use `.dropna()` when the missing field cannot be guessed at all, for example a row with no `funding_date` and no `amount`, which is not usable for any funding analysis anyway.

In [4]:
# ========== Q4 ==========

**Q4. pd.to_numeric(col, errors='coerce')**

Plain `pd.to_numeric(col)` raises a `ValueError` the moment it meets a value it cannot convert, such as the text `'Undisclosed'`. `errors='coerce'` converts whatever it can and turns anything that fails into `NaN` instead of stopping the whole operation. On real-world messy data, such as our `amount` column that mixes numbers with text like `'Undisclosed'` or currency symbols, `coerce` is the right choice, because it lets the rest of the pipeline run and leaves the bad values as `NaN` to be handled deliberately afterward.

In [5]:
# ========== Q5 ==========

**Q5. .loc vs .iloc**

`.loc` selects by label - `df.loc[3, 'amount']` gets the row whose index label is `3`, whatever that label happens to be. `.iloc` selects by integer position, so `df.iloc[0]` always returns the physically first row regardless of its label. They return different rows once the index is no longer a simple `0, 1, 2, ...` sequence - for example, after filtering a DataFrame without resetting the index, if the row originally labelled `0` was removed, `df.loc[0]` raises a `KeyError`, while `df.iloc[0]` still happily returns whatever row is now first in the table.

In [6]:
# ========== Q6 ==========

**Q6. .map() vs .apply() vs .replace()**

`.map()` only works on a Series and is meant for simple element-wise substitution, usually through a dict or a small function - good for mapping city aliases to a clean name. `.apply()` is more general: it works on a Series or a DataFrame, accepts more complex functions including lambdas with conditional logic, and on a DataFrame can run row-wise or column-wise. `.replace()` is meant purely for substituting specific matching values (a single value or a list) on either a Series or a DataFrame, without needing any function at all. I would reach for `.map()` for a clean lookup table, `.replace()` for a handful of exact value swaps, and `.apply()` when the transformation needs actual logic.

In [7]:
# ========== Q7 ==========

**Q7. SettingWithCopyWarning**

This warning appears when pandas cannot tell whether the object you are modifying is an independent copy or just a view into the original DataFrame, so it warns that your edit might silently not stick. A classic trigger:

```python
df_filtered = df[df['rating'] > 4]
df_filtered['rating'] = df_filtered['rating'] * 2
```

`df_filtered` may just be a view into `df`, so pandas cannot guarantee this assignment behaves as expected. The fix is to make the copy explicit:

```python
df_filtered = df[df['rating'] > 4].copy()
df_filtered['rating'] = df_filtered['rating'] * 2
```

The `.copy()` removes the ambiguity, so pandas knows this is a fully separate object and the warning disappears.

In [8]:
# ========== Q8 ==========

**Q8. pd.cut() vs pd.qcut()**

`pd.cut()` bins values into ranges that you define yourself, either equal-width by default or custom edges, so it is driven by the *value* boundaries. `pd.qcut()` bins values into groups with roughly equal *counts* of observations, so the bin edges shift depending on the data's distribution. For labelling customers as Low, Mid, or High earners by annual income, `pd.cut()` fits better, because those labels usually correspond to meaningful, fixed income thresholds that a business defines (say below 5 lakh, 5-15 lakh, above 15 lakh), not to whichever third of the customers happen to fall into a bucket that particular month.

## Section B - Predict the Output

In [9]:
# ========== Q9 ==========
# Prediction: df2 = df makes df2 point to the SAME object as df (no copy is made),
# so mutating df2['a'] also changes df. Expect df to show a = [10, 20, 30].
import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
df2 = df
df2['a'] = [10, 20, 30]
print(df)

    a  b
0  10  4
1  20  5
2  30  6


Prediction was correct - `df2 = df` copies the reference, not the data, so both names point to the same underlying object in memory.

In [10]:
# ========== Q10 ==========
# Prediction: .mean() skips NaN by default, so mean = (1+2+4+5)/4 = 3.0.
# .sum() also skips NaN by default, so sum = 1+2+4+5 = 12.0.
import numpy as np
s = pd.Series([1, 2, np.nan, 4, 5])
print(s.mean())
print(s.sum())

3.0
12.0


Prediction was correct - both `.mean()` and `.sum()` use `skipna=True` by default, so the `NaN` is simply excluded from the calculation rather than treated as zero.

In [11]:
# ========== Q11 ==========
# Prediction: 'NA' cannot be parsed as a number, so with errors='coerce' it becomes NaN.
# dtype becomes float64 (NaN forces a float column), sum ignores NaN -> 100+200+400 = 700.0.
df = pd.DataFrame({'price': ['100', '200', 'NA', '400']})
df['price'] = pd.to_numeric(df['price'], errors='coerce')
print(df['price'].dtype)
print(df['price'].sum())

float64
700.0


Prediction was correct.

In [12]:
# ========== Q12 ==========
# Prediction: df.loc[0] returns a single row collapsed into a Series, with the
# column names becoming the Series' index.
df = pd.DataFrame({'name': ['Rahul', 'Priya'], 'age': [22, 21]})
print(df.loc[0])
print(type(df.loc[0]))

name    Rahul
age        22
Name: 0, dtype: object
<class 'pandas.core.series.Series'>


Prediction was correct - selecting a single row from a DataFrame always collapses it to a `Series`.

In [13]:
# ========== Q13 ==========
# Prediction: df_filtered is a slice of df, so the assignment line raises a
# SettingWithCopyWarning. Whether df or df_filtered actually changes is not
# guaranteed by pandas - that ambiguity is exactly what the warning is about.
df = pd.DataFrame({'rating': [4.5, 3.8, 4.2, 5.0]})
df_filtered = df[df['rating'] > 4]
df_filtered['rating'] = df_filtered['rating'] * 2   # what warning?
print(df)
print(df_filtered)

   rating
0     4.5
1     3.8
2     4.2
3     5.0
   rating
0     9.0
2     8.4
3    10.0


/tmp/ipykernel_2419/2899913864.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['rating'] = df_filtered['rating'] * 2   # what warning?


Prediction was correct on the warning - this line throws a `SettingWithCopyWarning` because `df_filtered` is a filtered slice of `df`, and pandas cannot be sure whether the assignment is happening on a view or an independent copy. In this run `df` stayed unchanged and `df_filtered` updated correctly, but that outcome is not guaranteed, which is exactly why the fix is `df_filtered = df[df['rating'] > 4].copy()`.

In [14]:
# ========== Q14 ==========
# Prediction: apply() returns a mix of squared numbers and the string 'small'
# in the same Series, so pandas cannot keep a numeric dtype and falls back to
# the generic 'object' dtype.
df = pd.DataFrame({'a': [1, 2, 3, 4]})
result = df['a'].apply(lambda x: x**2 if x > 2 else 'small')
print(result)
print(result.dtype)

0    small
1    small
2        9
3       16
Name: a, dtype: object
object


Prediction was correct - mixing strings and numbers in one Series forces pandas to store everything as generic Python objects rather than a numeric dtype.

In [15]:
# ========== Q15 ==========
# Prediction: nunique() counts distinct NON-null values -> 3 (Delhi, Mumbai, Bengaluru).
# len(unique()) counts distinct values INCLUDING NaN as its own entry -> 4.
# value_counts() lists the count of each non-null value, sorted descending, NaN dropped by default.
df = pd.DataFrame({'city': ['Delhi', 'Mumbai', 'Delhi', 'Bengaluru', None]})
print(df['city'].nunique())
print(len(df['city'].unique()))
print(df['city'].value_counts())

3
4
city
Delhi        2
Mumbai       1
Bengaluru    1
Name: count, dtype: int64


Prediction was correct - `nunique()` drops `NaN` by default, `unique()` keeps `NaN` as one of the returned values (which is why `len()` is one higher), and `value_counts()` also drops `NaN` unless told otherwise with `dropna=False`.

In [16]:
# ========== Q16 ==========
# Prediction: the first print works fine and shows rows where a > 1.
# The commented line fails because '&' binds tighter than '>' in Python, so
# without parentheses it is evaluated as df['a'] > (1 & df['b']) > 15, which
# is not a valid boolean mask and raises a TypeError.
# The fix is to wrap each condition in its own parentheses, as in the last line.
df = pd.DataFrame({'a': [1, 2, 3], 'b': [10, 20, 30]})
print(df[df['a'] > 1])
# print(df[df['a'] > 1 & df['b'] > 15])   # <-- what's wrong here?
print(df[(df['a'] > 1) & (df['b'] > 15)])

   a   b
1  2  20
2  3  30
   a   b
1  2  20
2  3  30


Prediction was correct on the concept - the commented line breaks because of Python operator precedence: `&` binds tighter than `>`, so `1 & df['b']` gets evaluated before the comparisons, which does not produce a valid boolean mask and raises a `TypeError`. Always wrap each condition in its own parentheses when combining boolean masks with `&` or `|`.

## Section C - Applied on Indian Startups Dataset

All cleaning is done once, in the cell below, and the resulting `df` is reused for every question that follows.

In [17]:
# ========== Cleaning (done once, reused everywhere below) ==========
import pandas as pd
import numpy as np

df_raw = pd.read_csv("indian_startups_funding.csv")
df = df_raw.copy()

# 1. rename columns to clean snake_case
df.columns = ['startup_name', 'city', 'industry', 'funding_stage',
              'amount', 'funding_date', 'investors', 'founded_year']

# 2. clean startup_name and city - strip whitespace, title case
df['startup_name'] = df['startup_name'].str.strip().str.title()
df['city'] = df['city'].str.strip().str.title()

# 3. standardise city aliases (after .title(), so 'BLR' -> 'Blr' etc.)
city_map = {
    'Bangalore': 'Bengaluru', 'Blr': 'Bengaluru',
    'Bombay': 'Mumbai', 'Bom': 'Mumbai',
    'Gurgaon': 'Gurugram', 'Ggn': 'Gurugram',
    'Delhi': 'New Delhi',
    'Calcutta': 'Kolkata',
    'Madras': 'Chennai',
    'Hyd': 'Hyderabad',
}
df['city'] = df['city'].replace(city_map)

# 4a. standardise industry variants using a dict
df['industry'] = df['industry'].str.strip().str.title()
industry_map = {
    'Agritech': 'AgriTech', 'Agri Tech': 'AgriTech', 'Agri-Tech': 'AgriTech',
    'Ai Ml': 'AI/ML', 'Ai-Ml': 'AI/ML', 'Aiml': 'AI/ML', 'Ai/Ml': 'AI/ML',
    'Beautytech': 'BeautyTech', 'Beauty-Tech': 'BeautyTech',
    'E-Commerce': 'Ecommerce', 'Ecommerce': 'Ecommerce',
    'Edtech': 'EdTech', 'Ed-Tech': 'EdTech',
    'Ev': 'Electric Vehicles', 'Electric-Vehicles': 'Electric Vehicles',
    'Fintech': 'FinTech', 'Fin-Tech': 'FinTech', 'Ft': 'FinTech',
    'Food Tech': 'FoodTech', 'Food-Tech': 'FoodTech', 'Foodtech': 'FoodTech',
    'Games': 'Gaming',
    'Healthtech': 'HealthTech', 'Health Tech': 'HealthTech', 'Health-Tech': 'HealthTech',
    'Proptech': 'PropTech', 'Prop-Tech': 'PropTech',
    'Saas': 'SaaS',
    'Traveltech': 'TravelTech', 'Travel-Tech': 'TravelTech',
}
df['industry'] = df['industry'].replace(industry_map)

# 4b. standardise funding_stage variants using a dict
df['funding_stage'] = df['funding_stage'].str.strip().str.title()
stage_map = {
    'Ipo': 'IPO', 'Pre-Ipo': 'Pre-IPO',
    'Pre Seed': 'Pre-Seed', 'Preseed': 'Pre-Seed',
    'Pre Series A': 'Pre-Series A', 'Pre-A': 'Pre-Series A',
    'Series-A': 'Series A', 'Series_A': 'Series A',
    'Series-B': 'Series B', 'Series_B': 'Series B',
    'Series-C': 'Series C',
    'Series-D': 'Series D',
    'Series-E': 'Series E',
}
df['funding_stage'] = df['funding_stage'].replace(stage_map)

# 5. convert amount to numeric (in INR Cr): strip currency text, then coerce
amt = df['amount'].astype(str)
amt = (amt.str.replace('\u20b9', '', regex=False)   # rupee symbol
          .str.replace('INR', '', regex=False)
          .str.replace('Crore', '', regex=False)
          .str.replace('Cr', '', regex=False)
          .str.replace(',', '', regex=False)
          .str.strip())
df['amount'] = pd.to_numeric(amt, errors='coerce')   # 'Undisclosed' -> NaN

# 6. convert funding_date to datetime - the column mixes 4 different formats
#    (DD/MM/YYYY, DD-MM-YYYY, "DD Mon YYYY", YYYY/MM/DD), so a single fixed
#    format/dayfirst call cannot parse all rows. We try each known format in
#    turn and fall back to a dayfirst-coerced guess only if none of them match.
def parse_date(value):
    if pd.isna(value):
        return pd.NaT
    value = str(value).strip()
    for fmt in ('%d/%m/%Y', '%d-%m-%Y', '%d %b %Y', '%Y/%m/%d'):
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            continue
    return pd.to_datetime(value, errors='coerce', dayfirst=True)

df['funding_date'] = df['funding_date'].apply(parse_date)

# 7. drop duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("Cleaned shape:", df.shape)
df.head()

Cleaned shape: (450, 8)


,startup_name,city,industry,funding_stage,amount,funding_date,investors,founded_year
0,Playtime Games,New Delhi,Ecommerce,Series E,942.0,2024-02-29,"Ranjan Pai, Prosus Ventures",2020.0
1,Rebel Foods,Ahmedabad,BeautyTech,Pre-Series A,1440.0,2018-08-18,Lightspeed,2017.0
2,Freshworks,Kolkata,AgriTech,Pre-Series A,329.0,2020-07-14,Info Edge,2020.0
3,Bigbasket,Mumbai,Ecommerce,Series D,1405.0,2018-03-31,B Capital,2016.0
4,Ola,Kolkata,EdTech,Series E,2223.0,2021-08-24,Falcon Edge,2017.0


### Loading & Inspection (Q17-Q19)

In [18]:
# ========== Q17 ==========
# Stats computed on the RAW data, before any cleaning
print("(a) rows, columns:", df_raw.shape)
print()
print("(b) missing values per column:")
print(df_raw.isna().sum())
print()
print("(c) exact duplicate rows:", df_raw.duplicated().sum())

(a) rows, columns: (468, 8)

(b) missing values per column:
Startup Name      0
City              0
INDUSTRY          0
Funding Stage     0
Amount            0
Funding Date     19
Investors        10
Founded Year     23
dtype: int64

(c) exact duplicate rows: 18


In [19]:
# ========== Q18 ==========
print(sorted(df.columns))

['amount', 'city', 'founded_year', 'funding_date', 'funding_stage', 'industry', 'investors', 'startup_name']


In [20]:
# ========== Q19 ==========
fy = df['founded_year'].dropna()
stats = {
    'count': int(fy.count()),
    'mean': round(fy.mean(), 2),
    'min': fy.min(),
    'max': fy.max(),
    'median': fy.median(),
    'std': round(fy.std(), 2),
}
print(stats)

{'count': 427, 'mean': np.float64(2016.43), 'min': 2008.0, 'max': 2022.0, 'median': 2017.0, 'std': 4.25}


### Cleaning Verification (Q20-Q24)

In [21]:
# ========== Q20 ==========
cities = sorted(df['city'].unique())
print(cities)
print("Number of unique cities:", len(cities))

['Ahmedabad', 'Bengaluru', 'Chennai', 'Gurugram', 'Hyderabad', 'Kolkata', 'Mumbai', 'New Delhi', 'Noida', 'Pune']
Number of unique cities: 10


In [22]:
# ========== Q21 ==========
print(df['industry'].value_counts())

industry
AI/ML                54
FoodTech             39
SaaS                 36
BeautyTech           34
TravelTech           34
Gaming               34
PropTech             32
HealthTech           31
Ecommerce            30
AgriTech             29
Logistics            27
Electric Vehicles    27
FinTech              22
EdTech               21
Name: count, dtype: int64


In [23]:
# ========== Q22 ==========
print("(a) dtype:", df['amount'].dtype)
print("(b) NaN count:", df['amount'].isna().sum())
print("(c) mean funding amount:", round(df['amount'].mean(), 2))
print("(d) min:", df['amount'].min(), " max:", df['amount'].max())

(a) dtype: float64
(b) NaN count: 24
(c) mean funding amount: 753.06
(d) min: 1.0  max: 3549.0


In [24]:
# ========== Q23 ==========
deals_per_year = df['funding_date'].dt.year.value_counts().sort_index()
print("(a) deals per year:")
print(deals_per_year)
print()
print("(b) year with most funding deals:", df['funding_date'].dt.year.value_counts().idxmax())

(a) deals per year:
funding_date
2008.0     2
2009.0     1
2010.0     7
2011.0     4
2012.0     7
2013.0     7
2014.0     7
2015.0     8
2016.0    21
2017.0    21
2018.0    25
2019.0    43
2020.0    39
2021.0    59
2022.0    70
2023.0    69
2024.0    42
Name: count, dtype: int64

(b) year with most funding deals: 2022.0


In [25]:
# ========== Q24 ==========
print(df['funding_stage'].value_counts())

funding_stage
Series B        56
Pre-Seed        48
Series E        48
Seed            47
Series C        46
Series A        43
Bridge          41
Series D        38
Pre-Series A    38
IPO             33
Pre-IPO         12
Name: count, dtype: int64


### Filtering (Q25-Q28)

In [26]:
# ========== Q25 ==========
q25 = (df[(df['city'] == 'Bengaluru') & (df['amount'] > 100)]
       .sort_values('amount', ascending=False)[['startup_name', 'amount']])
print(q25)

       startup_name  amount
301        Routemap  3528.0
223     Agriconnect  3526.0
318     Whitehat Jr  3511.0
152        Ecomotor  2244.0
314       Beautybox  2239.0
98           Meesho  2231.0
144       Curefoods  2230.0
26      Mediconnect  2226.0
292     Villagemart  2195.0
233       Vitalcare  1431.0
272           Paytm  1423.0
65        Instaloan  1420.0
247  Playtime Games  1410.0
355            Cred   950.0
67      Simplilearn   923.0
222    Smartscholar   899.0
336         Flyhigh   728.0
431        Agroplus   726.0
285     Agriconnect   713.0
240        Stylehub   705.0
250      Harvestify   497.0
330          Practo   486.0
230       Vitalcare   456.0
88            Zepto   455.0
387        Shopeasy   449.0
147     Mediconnect   304.0
85        Smartshop   214.0
40         Rupaypro   186.0
180         Fetchit   150.0
419       Games24X7   118.0
384         Cuemath   115.0


In [27]:
# ========== Q26 ==========
q26 = df[df['industry'].isin(['FinTech', 'EdTech', 'HealthTech'])]
print("count:", len(q26))

count: 74


In [28]:
# ========== Q27 ==========
# founded_year has missing values - .between() simply returns False for NaN
# rows, which is exactly the behaviour we want here (they should not be counted).
q27 = df[df['founded_year'].between(2015, 2020)]
print("count:", len(q27))

count: 211


In [29]:
# ========== Q28 ==========
q28 = df[df['startup_name'].str.contains('pay', case=False, na=False)]['startup_name'].unique()
print(q28)

['Rupaypro' 'Razorpay' 'Paytm' 'Paybuddy' 'Quickpay India']


### Transformation & Derived Columns (Q29-Q32)

In [30]:
# ========== Q29 ==========
df['funding_efficiency'] = df['amount'] / (2024 - df['founded_year'])
q29 = (df.sort_values('funding_efficiency', ascending=False)
         .head(10)[['startup_name', 'industry', 'amount', 'founded_year', 'funding_efficiency']])
print(q29)

    startup_name           industry  amount  founded_year  funding_efficiency
311   Classroomx              AI/ML  3520.0        2022.0         1760.000000
205     Cropcare          Ecommerce  3518.0        2021.0         1172.666667
121        Nykaa          Logistics  3515.0        2021.0         1171.666667
152     Ecomotor             Gaming  2244.0        2022.0         1122.000000
156   Makemytrip  Electric Vehicles  2202.0        2022.0         1101.000000
55       Blinkit  Electric Vehicles  2196.0        2022.0         1098.000000
243  Fitnessplus            FinTech  3545.0        2020.0          886.250000
169     Stayeasy          Ecommerce  2249.0        2021.0          749.666667
314    Beautybox          Ecommerce  2239.0        2021.0          746.333333
432   Cleanslate  Electric Vehicles  2204.0        2021.0          734.666667


In [31]:
# ========== Q30 ==========
def tier(amount):
    if pd.isna(amount):
        return 'Small / Unknown'
    if amount >= 1000:
        return 'Unicorn'
    if amount >= 500:
        return 'Mega'
    if amount >= 100:
        return 'Large'
    if amount >= 10:
        return 'Mid'
    return 'Small / Unknown'

df['funding_tier'] = df['amount'].apply(lambda x: tier(x))
print(df['funding_tier'].value_counts())

funding_tier
Large              131
Mid                120
Unicorn             97
Mega                68
Small / Unknown     34
Name: count, dtype: int64


In [32]:
# ========== Q31 ==========
df['era'] = pd.cut(df['founded_year'],
                    bins=[0, 2009, 2017, 3000],
                    labels=['Early', 'Growth', 'Recent'])
print(df['era'].value_counts())

era
Recent    207
Growth    195
Early      25
Name: count, dtype: int64


In [33]:
# ========== Q32 ==========
df['investor_count'] = df['investors'].apply(
    lambda x: 0 if pd.isna(x) else len(str(x).split(', '))
)
print(df['investor_count'].value_counts().sort_index())

investor_count
0     10
1     94
2    199
3    118
4     29
Name: count, dtype: int64


### Aggregation & Grouping (Q33-Q37)

In [34]:
# ========== Q33 ==========
print(df.groupby('industry')['amount'].mean().sort_values(ascending=False).head(5))

industry
FinTech              1217.636364
EdTech               1191.250000
TravelTech           1115.968750
Electric Vehicles    1090.769231
Ecommerce            1035.107143
Name: amount, dtype: float64


In [35]:
# ========== Q34 ==========
city_agg = df.groupby('city').agg(
    num_startups=('startup_name', 'count'),
    total_funding=('amount', 'sum'),
    avg_funding=('amount', 'mean'),
)
print(city_agg.sort_values('num_startups', ascending=False).head(10))

           num_startups  total_funding  avg_funding
city                                               
Noida                62        41499.0   715.500000
Kolkata              51        34029.0   708.937500
Bengaluru            49        39315.0   836.489362
Mumbai               45        32089.0   782.658537
Ahmedabad            44        28250.0   642.045455
Gurugram             44        33275.0   792.261905
Hyderabad            43        29738.0   743.450000
Pune                 42        33333.0   854.692308
Chennai              37        20667.0   558.567568
New Delhi            33        28608.0   953.600000


In [36]:
# ========== Q35 ==========
ind_stats = df.groupby('industry')['amount'].agg(['count', 'mean'])
ind_filtered = ind_stats[ind_stats['count'] >= 15].sort_values('mean', ascending=False)
print(ind_filtered.head(1))

          count         mean
industry                    
FinTech      22  1217.636364


In [37]:
# ========== Q36 ==========
top5_ind = df['industry'].value_counts().head(5).index
sub = df[df['industry'].isin(top5_ind)]
ct = pd.crosstab(sub['industry'], sub['funding_tier'])
print(ct)

funding_tier  Large  Mega  Mid  Small / Unknown  Unicorn
industry                                                
AI/ML            12     7   20                6        9
BeautyTech       14     4    8                2        6
FoodTech         12     5   11                2        9
SaaS             11     4   11                3        7
TravelTech       11     7    4                2       10


In [38]:
# ========== Q37 ==========
idx = df.groupby('industry')['amount'].idxmax().dropna()
q37 = df.loc[idx, ['startup_name', 'industry', 'city', 'amount']]
print(q37)

    startup_name           industry       city  amount
311   Classroomx              AI/ML  Ahmedabad  3520.0
193    Boldfoods           AgriTech  Hyderabad  1449.0
93     Vitalcare         BeautyTech       Pune  3522.0
168      Curefit          Ecommerce     Mumbai  3540.0
322    Fitfusion             EdTech     Mumbai  3546.0
116        Paytm  Electric Vehicles       Pune  3533.0
243  Fitnessplus            FinTech  New Delhi  3545.0
21   Simplilearn           FoodTech    Kolkata  3549.0
177    Wealthwiz             Gaming    Kolkata  3549.0
372   Investedge         HealthTech    Kolkata  2219.0
121        Nykaa          Logistics  New Delhi  3515.0
190    Mamaearth           PropTech  Hyderabad  2235.0
56   Whitehat Jr               SaaS  Hyderabad  2250.0
28      Rupaypro         TravelTech  Ahmedabad  3541.0


### Real Industry Problems (Q38-Q40)

In [39]:
# ========== Q38 ==========
emerging = df[(df['founded_year'] >= 2019) &
              (df['amount'] >= 100) &
              (df['industry'].isin(['FinTech', 'EdTech', 'HealthTech']))]
emerging = emerging.sort_values('amount', ascending=False)[
    ['startup_name', 'industry', 'city', 'founded_year', 'amount']
]
print(emerging)

      startup_name    industry       city  founded_year  amount
243    Fitnessplus     FinTech  New Delhi        2020.0  3545.0
57       Learnlive     FinTech      Noida        2019.0  3512.0
372     Investedge  HealthTech    Kolkata        2020.0  2219.0
176     Classroomx     FinTech      Noida        2022.0  1418.0
264        Vedantu  HealthTech      Noida        2021.0  1413.0
444        Postman      EdTech  Hyderabad        2019.0  1404.0
355           Cred      EdTech  Bengaluru        2020.0   950.0
331        Voltcar      EdTech  Hyderabad        2021.0   916.0
364  Bookmyservice  HealthTech      Noida        2021.0   902.0
402    Villagemart      EdTech   Gurugram        2021.0   896.0
356      Agriboost     FinTech  New Delhi        2021.0   729.0
433      Groomguru  HealthTech      Noida        2021.0   722.0
128      Dermaplus  HealthTech   Gurugram        2019.0   711.0
329       Fundflow     FinTech      Noida        2019.0   709.0
323       Shopeasy     FinTech   Gurugra

A VC analyst would care about this list because it is effectively a shortlist of young companies (founded 2019 or later) that have already raised serious capital in three of the hottest Indian sectors. Rather than scanning the whole market by hand, this filter surfaces which newer players are scaling fast enough to be worth a closer look ahead of their next funding round.

In [40]:
# ========== Q39 ==========
city_grp = df.groupby('city').agg(count=('startup_name', 'count'), total_funding=('amount', 'sum'))
city_grp = city_grp[city_grp['count'] >= 20].sort_values('total_funding', ascending=False)
print(city_grp)

           count  total_funding
city                           
Noida         62        41499.0
Bengaluru     49        39315.0
Kolkata       51        34029.0
Pune          42        33333.0
Gurugram      44        33275.0
Mumbai        45        32089.0
Hyderabad     43        29738.0
New Delhi     33        28608.0
Ahmedabad     44        28250.0
Chennai       37        20667.0


In [41]:
# ========== Q40 ==========
sat = (df.groupby(['city', 'industry']).size()
         .reset_index(name='count')
         .sort_values('count', ascending=False)
         .head(10))
print(sat)

          city    industry  count
46    Gurugram    FoodTech     10
21   Bengaluru      Gaming      8
79      Mumbai       AI/ML      8
120       Pune       AI/ML      8
0    Ahmedabad       AI/ML      7
65     Kolkata       AI/ML      7
77     Kolkata        SaaS      7
113      Noida    FoodTech      7
119      Noida  TravelTech      7
14   Bengaluru    AgriTech      7


Gurugram paired with FoodTech is the single most crowded city-industry combination in the dataset, so a new founder eyeing that exact pairing should expect heavy competition. The more useful read for a founder is the reverse: city-industry pairs that never show up in this top-10 list are the ones with the most room to move.

---
### A note on AI assistance
I used AI assistance to help structure and verify the cleaning pipeline (particularly the multi-format date parsing in the cleaning cell and a couple of the groupby/agg patterns in Section C), and to sanity-check my Section B predictions before running the cells. However all theory answers, the choice of mapping dictionaries, and the interpretation write-ups for Q38 and Q40 are my own.